# LivePortrait GStreamer Plugin Tutorial

This notebook demonstrates how to use the `gst-liveportrait` plugin for real-time head reenactment using our pre-built Docker environment. 

> **Note:** Output videos now include robust audio and support for dynamic pose augmentation.

## 1. Prerequisites

Ensure you have the TensorRT engines exported. You will need Docker and NVIDIA Container Toolkit installed on your host machine.

## 2. Compile the Plugin

Compile the C++ source code inside the container for your specific GPU architecture.

In [ ]:
!docker run --rm -v $(pwd)/..:/workspace -w /workspace ducksouplab/liveportrait_gst:latest bash -c "mkdir -p build && cd build && cmake .. && make -j$(nproc)"

## 3. Basic Reenactment (with Audio)

The `liveportrait_process.py` wrapper automatically muxes audio from the driving video into the output.

In [ ]:
# Reenacting Petter
!python3 ../liveportrait_process.py \
    --input ../assets/video_example.mp4 \
    --output ../outputs/petter_std.mp4 \
    --source ../assets/Petter-Johansson.jpg \
    --config ../checkpoints/

## 4. Head Pose Augmentation (Nodding)

You can inject relative head movements on top of the natural driving motion using `--enable-pose-offset`.

### Example: Programmatic Nodding
For dynamic movement (like a sine-wave nod), you can use the provided `nod_demo.py` script which uses the GStreamer Python API.

In [ ]:
# Generate a dynamic nodding video for test_image.jpg
!docker run --rm --gpus all \
    -v $(pwd)/..:/workspace \
    -v $(pwd)/../assets:/work \
    -v $(pwd)/../assets:/source \
    -v $(pwd)/../checkpoints:/checkpoints \
    -v $(pwd)/../build:/plugin \
    -v $(pwd)/../outputs:/outputs \
    -e GST_PLUGIN_PATH=/plugin \
    -w /workspace \
    ducksouplab/liveportrait_gst:latest \
    python3 nod_demo.py --source /source/test_image.jpg --output /outputs/test1_nod.mp4

## 5. Side-by-Side Comparison

Generate a comparison video showing the driving source and the reenactment side-by-side.

In [ ]:
!python3 ../liveportrait_process.py \
    --input ../assets/video_example.mp4 \
    --output ../outputs/test1_side_by_side.mp4 \
    --source ../assets/test_image.jpg \
    --config ../checkpoints/ \
    --side-by-side